In [ ]:
from sklearn.metrics import mean_squared_error


In [ ]:
# Input dimensions
input_dim = 4  # Match the number of features in the dataset

# Build generator and discriminator
generator = build_generator(input_dim)
discriminator = build_discriminator(input_dim)

# Compile discriminator
discriminator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# GAN combined model
discriminator.trainable = False
gan_input = layers.Input(shape=(input_dim,))
generated_data = generator(gan_input)
gan_output = discriminator(generated_data)
gan = tf.keras.Model(gan_input, gan_output)
gan.compile(optimizer='adam', loss='binary_crossentropy')

# Training the GAN
def train_gan(generator, discriminator, gan, data_with_gaps, mask, epochs=1000, batch_size=5):  # Adjust batch_size
    for epoch in range(epochs):
        # Generate fake data
        noise = tf.random.normal((batch_size, input_dim))
        generated_data = generator.predict(noise)

        # Create datasets for discriminator
        real_data = data_with_gaps[:batch_size]
        fake_data = generated_data
        X_discriminator = tf.concat([real_data, fake_data], axis=0)
        y_discriminator = tf.concat([tf.ones((batch_size, 1)), tf.zeros((batch_size, 1))], axis=0)

        # Train discriminator
        discriminator.trainable = True
        discriminator_loss = discriminator.train_on_batch(X_discriminator, y_discriminator)

        # Train generator
        noise = tf.random.normal((batch_size, input_dim))
        y_generator = tf.ones((batch_size, 1))
        discriminator.trainable = False
        generator_loss = gan.train_on_batch(noise, y_generator)

        # Print losses
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Generator Loss: {generator_loss}, Discriminator Loss: {discriminator_loss}")

# Example usage
train_gan(generator, discriminator, gan, data_with_gaps, mask)


In [ ]:
def calculate_rmse(generator, data_with_gaps, mask, batch_size=5):
    # Generate data for RMSE calculation
    noise = tf.random.normal((data_with_gaps.shape[0], input_dim))
    generated_data = generator.predict(noise)

    # Extract real and generated values where mask == 1
    real_missing_values = data_with_gaps[mask == 1]
    imputed_values = generated_data[mask == 1]

    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(real_missing_values, imputed_values))
    return rmse

# Call this function after training
rmse = calculate_rmse(generator, data_with_gaps, mask)
print(f"RMSE after GAN training: {rmse}")

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import mean_squared_error

df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 3horas.csv')





# Drop unnecessary columns (other columns were already excluded in the provided dataset)
df = df[["Timestamp_cubic", "Vazao"]]

# Remove 10% of Vazao data
np.random.seed(42)
mask_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
original_vazao = df["Vazao"].copy()  # Backup the original data for RMSE calculation
df.loc[mask_indices, "Vazao"] = np.nan

# Create mask for missing data
mask = df["Vazao"].isnull().astype(int).values.reshape(-1, 1)

# Replace NaN with placeholders (e.g., 0.0)
data_with_gaps = df["Vazao"].fillna(0).values.reshape(-1, 1)

# GAN Implementation
input_dim = 1  # Single feature (Vazao)

# Build Generator
def build_generator(input_dim):
    model = tf.keras.Sequential([
        layers.Dense(128, activation="relu", input_dim=input_dim),
        layers.Dense(128, activation="relu"),
        layers.Dense(input_dim, activation="linear")
    ])
    return model

# Build Discriminator
def build_discriminator(input_dim):
    model = tf.keras.Sequential([
        layers.Dense(128, activation="relu", input_dim=input_dim),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

# Initialize Generator and Discriminator
generator = build_generator(input_dim)
discriminator = build_discriminator(input_dim)

# Compile Discriminator
discriminator.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Combine GAN
discriminator.trainable = False
gan_input = layers.Input(shape=(input_dim,))
generated_data = generator(gan_input)
gan_output = discriminator(generated_data)
gan = tf.keras.Model(gan_input, gan_output)
gan.compile(optimizer="adam", loss="binary_crossentropy")

# GAN Training
 # GAN Training
def train_gan(generator, discriminator, gan, data_with_gaps, mask, epochs=5000, batch_size=4):
    for epoch in range(epochs):
        # Generate fake data
        noise = tf.random.normal((batch_size, input_dim))
        generated_data = generator.predict(noise, verbose=0)

        # Create datasets for discriminator
        real_data = data_with_gaps[:batch_size]
        fake_data = generated_data
        X_discriminator = np.vstack((real_data, fake_data))  # Combine real and fake data
        y_discriminator = np.vstack((np.ones((batch_size, 1)), np.zeros((batch_size, 1))))  # Correctly stack labels

        # Train discriminator
        discriminator.trainable = True
        discriminator_loss = discriminator.train_on_batch(X_discriminator, y_discriminator)

        # Train generator
        noise = tf.random.normal((batch_size, input_dim))
        y_generator = np.ones((batch_size, 1))
        discriminator.trainable = False
        generator_loss = gan.train_on_batch(noise, y_generator)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}: Generator Loss = {generator_loss}, Discriminator Loss = {discriminator_loss[0]}")


# Train the GAN
train_gan(generator, discriminator, gan, data_with_gaps, mask)

# Imputation with GAN
noise = tf.random.normal((len(data_with_gaps), input_dim))
generated_data = generator.predict(noise, verbose=0)

# Replace missing values with generated data
imputed_data = data_with_gaps.copy()
imputed_data[mask.flatten() == 1] = generated_data[mask.flatten() == 1]

# Calculate RMSE
real_missing_values = original_vazao[mask.flatten() == 1]
imputed_values = imputed_data[mask.flatten() == 1]
rmse = np.sqrt(mean_squared_error(real_missing_values, imputed_values))

print(f"RMSE: {rmse/1000000}")


In [ ]:
# InfoGAN Implementation
from tensorflow.keras.models import Model

# Generator for InfoGAN
def build_infogan_generator(input_dim, noise_dim, condition_dim):
    model = tf.keras.Sequential([
        layers.InputLayer(input_shape=(noise_dim + condition_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(input_dim, activation="linear")
    ])
    return model

# Discriminator for InfoGAN
def build_infogan_discriminator(input_dim, condition_dim):
    input_data = layers.Input(shape=(input_dim,))
    input_condition = layers.Input(shape=(condition_dim,))
    x = layers.concatenate([input_data, input_condition])
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    validity = layers.Dense(1, activation="sigmoid")(x)
    model = Model([input_data, input_condition], validity)
    return model

# InfoGAN Generator and Discriminator
noise_dim = 1
condition_dim = 1  # Use the timestamp or some feature as the condition
infogan_generator = build_infogan_generator(input_dim, noise_dim, condition_dim)
infogan_discriminator = build_infogan_discriminator(input_dim, condition_dim)

# Compile Discriminator
infogan_discriminator.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Combine InfoGAN
noise_input = layers.Input(shape=(noise_dim,))
condition_input = layers.Input(shape=(condition_dim,))
generated_data = infogan_generator(layers.concatenate([noise_input, condition_input]))
validity = infogan_discriminator([generated_data, condition_input])
infogan = Model([noise_input, condition_input], validity)
infogan.compile(optimizer="adam", loss="binary_crossentropy")

# Training InfoGAN
def train_infogan(generator, discriminator, gan, data_with_gaps, mask, epochs=5000, batch_size=4):
    for epoch in range(epochs):
        # Generate fake data
        noise = tf.random.normal((batch_size, noise_dim))
        conditions = np.random.uniform(0, 1, (batch_size, condition_dim))  # Randomized conditions
        generated_data = generator.predict(np.concatenate([noise, conditions], axis=1), verbose=0)

        # Prepare datasets for discriminator
        real_data = data_with_gaps[:batch_size]
        fake_data = generated_data
        real_conditions = np.random.uniform(0, 1, (batch_size, condition_dim))  # Simulated conditions
        X_discriminator = [np.vstack((real_data, fake_data)), np.vstack((real_conditions, conditions))]
        y_discriminator = np.vstack((np.ones((batch_size, 1)), np.zeros((batch_size, 1))))

        # Train discriminator
        discriminator.trainable = True
        discriminator_loss = discriminator.train_on_batch(X_discriminator, y_discriminator)

        # Train generator
        noise = tf.random.normal((batch_size, noise_dim))
        conditions = np.random.uniform(0, 1, (batch_size, condition_dim))
        y_generator = np.ones((batch_size, 1))
        discriminator.trainable = False
        generator_loss = gan.train_on_batch([noise, conditions], y_generator)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}: Generator Loss = {generator_loss}, Discriminator Loss = {discriminator_loss[0]}")

# Train InfoGAN
train_infogan(infogan_generator, infogan_discriminator, infogan, data_with_gaps, mask)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error

# Carregar o dataset
df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 3horas.csv')

# Selecionar as colunas necessárias
df = df[["Timestamp_cubic", "Vazao"]]

# Remover 10% dos dados de Vazao
np.random.seed(42)
mask_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
original_vazao = df["Vazao"].copy()  # Backup dos valores reais para cálculo do RMSE
df.loc[mask_indices, "Vazao"] = np.nan

# Criar uma máscara para identificar os dados ausentes
mask = df["Vazao"].isnull().astype(int).values.reshape(-1, 1)

# Substituir os valores ausentes usando KNN
imputer = KNNImputer(n_neighbors=3)  # Número de vizinhos pode ser ajustado
data_with_knn = df[["Vazao"]].values  # Selecionar apenas a coluna Vazao
imputed_data_knn = imputer.fit_transform(data_with_knn)

# Cálculo do RMSE com KNN
real_missing_values = original_vazao[mask.flatten() == 1]
imputed_values_knn = imputed_data_knn[mask.flatten() == 1]
rmse_knn = np.sqrt(mean_squared_error(real_missing_values, imputed_values_knn))

print(f"RMSE (KNN): {rmse_knn/1000000}")


In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import mean_squared_error

df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 3horas.csv')





# Drop unnecessary columns (other columns were already excluded in the provided dataset)
df = df[["Timestamp_cubic", "Vazao_bbr"]]

# Remove 10% of Vazao data
np.random.seed(42)
mask_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
original_vazao = df["Vazao_bbr"].copy()  # Backup the original data for RMSE calculation
df.loc[mask_indices, "Vazao_bbr"] = np.nan

# Create mask for missing data
mask = df["Vazao_bbr"].isnull().astype(int).values.reshape(-1, 1)

# Replace NaN with placeholders (e.g., 0.0)
data_with_gaps = df["Vazao_bbr"].fillna(0).values.reshape(-1, 1)

# GAN Implementation
input_dim = 1  # Single feature (Vazao)

# Build Generator
def build_generator(input_dim):
    model = tf.keras.Sequential([
        layers.Dense(128, activation="relu", input_dim=input_dim),
        layers.Dense(128, activation="relu"),
        layers.Dense(input_dim, activation="linear")
    ])
    return model

# Build Discriminator
def build_discriminator(input_dim):
    model = tf.keras.Sequential([
        layers.Dense(128, activation="relu", input_dim=input_dim),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

# Initialize Generator and Discriminator
generator = build_generator(input_dim)
discriminator = build_discriminator(input_dim)

# Compile Discriminator
discriminator.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Combine GAN
discriminator.trainable = False
gan_input = layers.Input(shape=(input_dim,))
generated_data = generator(gan_input)
gan_output = discriminator(generated_data)
gan = tf.keras.Model(gan_input, gan_output)
gan.compile(optimizer="adam", loss="binary_crossentropy")

# GAN Training
 # GAN Training
def train_gan(generator, discriminator, gan, data_with_gaps, mask, epochs=5000, batch_size=4):
    for epoch in range(epochs):
        # Generate fake data
        noise = tf.random.normal((batch_size, input_dim))
        generated_data = generator.predict(noise, verbose=0)

        # Create datasets for discriminator
        real_data = data_with_gaps[:batch_size]
        fake_data = generated_data
        X_discriminator = np.vstack((real_data, fake_data))  # Combine real and fake data
        y_discriminator = np.vstack((np.ones((batch_size, 1)), np.zeros((batch_size, 1))))  # Correctly stack labels

        # Train discriminator
        discriminator.trainable = True
        discriminator_loss = discriminator.train_on_batch(X_discriminator, y_discriminator)

        # Train generator
        noise = tf.random.normal((batch_size, input_dim))
        y_generator = np.ones((batch_size, 1))
        discriminator.trainable = False
        generator_loss = gan.train_on_batch(noise, y_generator)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}: Generator Loss = {generator_loss}, Discriminator Loss = {discriminator_loss[0]}")


# Train the GAN
train_gan(generator, discriminator, gan, data_with_gaps, mask)

# Imputation with GAN
noise = tf.random.normal((len(data_with_gaps), input_dim))
generated_data = generator.predict(noise, verbose=0)

# Replace missing values with generated data
imputed_data = data_with_gaps.copy()
imputed_data[mask.flatten() == 1] = generated_data[mask.flatten() == 1]

# Calculate RMSE
real_missing_values = original_vazao[mask.flatten() == 1]
imputed_values = imputed_data[mask.flatten() == 1]
rmse = np.sqrt(mean_squared_error(real_missing_values, imputed_values))

print(f"RMSE: {rmse/1000000}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error

# Carregar o dataset
df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 3horas.csv')

# Selecionar as colunas necessárias
df = df[["Timestamp_cubic", "Vazao_bbr"]]

# Remover 10% dos dados de Vazao
np.random.seed(42)
mask_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
original_vazao = df["Vazao_bbr"].copy()  # Backup dos valores reais para cálculo do RMSE
df.loc[mask_indices, "Vazao_bbr"] = np.nan

# Criar uma máscara para identificar os dados ausentes
mask = df["Vazao_bbr"].isnull().astype(int).values.reshape(-1, 1)

# Substituir os valores ausentes usando KNN
imputer = KNNImputer(n_neighbors=3)  # Número de vizinhos pode ser ajustado
data_with_knn = df[["Vazao_bbr"]].values  # Selecionar apenas a coluna Vazao
imputed_data_knn = imputer.fit_transform(data_with_knn)

# Cálculo do RMSE com KNN
real_missing_values = original_vazao[mask.flatten() == 1]
imputed_values_knn = imputed_data_knn[mask.flatten() == 1]
rmse_knn = np.sqrt(mean_squared_error(real_missing_values, imputed_values_knn))

print(f"RMSE (KNN): {rmse_knn/1000000}")


InfoGAN

In [ ]:
from tensorflow.keras.models import Model
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import mean_squared_error

# Load dataset
df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 3horas.csv')

# Select necessary columns
df = df[["Timestamp_cubic", "Vazao_bbr"]]

# Remove 10% of Vazao data
np.random.seed(42)
mask_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
original_vazao = df["Vazao_bbr"].copy()
df.loc[mask_indices, "Vazao_bbr"] = np.nan

# Create mask for missing data
mask = df["Vazao_bbr"].isnull().astype(int).values.reshape(-1, 1)
data_with_gaps = df["Vazao_bbr"].fillna(0).values.reshape(-1, 1)

# Normalize conditions (timestamps)
conditions = df["Timestamp_cubic"].values.reshape(-1, 1)
conditions = (conditions - conditions.min()) / (conditions.max() - conditions.min())

# Dimensions
input_dim = 1  # For Vazao_bbr
noise_dim = 1  # Noise input
condition_dim = 1  # Timestamp condition

# InfoGAN Generator
def build_infogan_generator(input_dim, noise_dim, condition_dim):
    input_layer = layers.Input(shape=(noise_dim + condition_dim,))
    x = layers.Dense(128, activation="relu")(input_layer)
    x = layers.Dense(128, activation="relu")(x)
    output_layer = layers.Dense(input_dim, activation="linear")(x)
    return Model(input_layer, output_layer)

# InfoGAN Discriminator
def build_infogan_discriminator(input_dim, condition_dim):
    data_input = layers.Input(shape=(input_dim,))
    condition_input = layers.Input(shape=(condition_dim,))
    merged_input = layers.concatenate([data_input, condition_input])
    x = layers.Dense(128, activation="relu")(merged_input)
    x = layers.Dense(128, activation="relu")(x)
    validity = layers.Dense(1, activation="sigmoid")(x)
    return Model([data_input, condition_input], validity)

# Initialize Generator and Discriminator
infogan_generator = build_infogan_generator(input_dim, noise_dim, condition_dim)
infogan_discriminator = build_infogan_discriminator(input_dim, condition_dim)

# Compile Discriminator
infogan_discriminator.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Combine InfoGAN
noise_input = layers.Input(shape=(noise_dim,))
condition_input = layers.Input(shape=(condition_dim,))
generated_data = infogan_generator(layers.concatenate([noise_input, condition_input]))
validity = infogan_discriminator([generated_data, condition_input])
infogan = Model([noise_input, condition_input], validity)
infogan.compile(optimizer="adam", loss="binary_crossentropy")

# Training InfoGAN
def train_infogan(generator, discriminator, gan, data_with_gaps, conditions, mask, epochs=5000, batch_size=64):
    for epoch in range(epochs):
        # Generate fake data
        noise = tf.random.normal((batch_size, noise_dim))
        real_indices = np.random.choice(len(data_with_gaps), batch_size, replace=False)
        real_data = data_with_gaps[real_indices]
        real_conditions = conditions[real_indices]

        generated_data = generator.predict(np.concatenate([noise, real_conditions], axis=1), verbose=0)

        # Prepare data for discriminator
        X_discriminator = [np.vstack((real_data, generated_data)),
                           np.vstack((real_conditions, real_conditions))]
        y_discriminator = np.vstack((np.ones((batch_size, 1)), np.zeros((batch_size, 1))))

        # Train discriminator
        discriminator.trainable = True
        discriminator_loss = discriminator.train_on_batch(X_discriminator, y_discriminator)

        # Train generator
        noise = tf.random.normal((batch_size, noise_dim))
        y_generator = np.ones((batch_size, 1))
        discriminator.trainable = False
        generator_loss = gan.train_on_batch([noise, real_conditions], y_generator)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}: Generator Loss = {generator_loss}, Discriminator Loss = {discriminator_loss[0]}")

# Train InfoGAN
train_infogan(infogan_generator, infogan_discriminator, infogan, data_with_gaps, conditions, mask)

# Imputation with InfoGAN
noise = tf.random.normal((len(data_with_gaps), noise_dim))
imputed_values = infogan_generator.predict(np.concatenate([noise, conditions], axis=1), verbose=0)
imputed_data = data_with_gaps.copy()
imputed_data[mask.flatten() == 1] = imputed_values[mask.flatten() == 1]

# Calculate RMSE
real_missing_values = original_vazao[mask.flatten() == 1]
rmse = np.sqrt(mean_squared_error(real_missing_values, imputed_data[mask.flatten() == 1]))
print(f"RMSE: {rmse}")


Spatio-Temporal GAN

In [ ]:
from tensorflow.keras.models import Model
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import mean_squared_error

# Carregar o dataset
df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 3horas.csv')

# Selecionar as colunas necessárias
df = df[["Timestamp_cubic", "Vazao_bbr"]]

# Remover 10% dos dados de Vazao
np.random.seed(42)
mask_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
original_vazao = df["Vazao_bbr"].copy()
df.loc[mask_indices, "Vazao_bbr"] = np.nan

# Criar uma máscara para identificar os dados ausentes
mask = df["Vazao_bbr"].isnull().astype(int).values.reshape(-1, 1)
data_with_gaps = df["Vazao_bbr"].fillna(0).values.reshape(-1, 1)

# Normalizar os timestamps como condições temporais
conditions = df["Timestamp_cubic"].values.reshape(-1, 1)
conditions = (conditions - conditions.min()) / (conditions.max() - conditions.min())  # Normalização

# Dimensões
input_dim = 1  # Para Vazao_bbr
time_dim = 1  # Para Timestamp_cubic

# Spatio-Temporal Generator
def build_stgan_generator(input_dim, time_dim):
    input_layer = layers.Input(shape=(input_dim + time_dim,))
    x = layers.Dense(128, activation="relu")(input_layer)
    x = layers.Dense(128, activation="relu")(x)
    output_layer = layers.Dense(input_dim, activation="linear")(x)
    return Model(input_layer, output_layer)

# Spatio-Temporal Discriminator
def build_stgan_discriminator(input_dim, time_dim):
    data_input = layers.Input(shape=(input_dim,))
    time_input = layers.Input(shape=(time_dim,))
    merged_input = layers.concatenate([data_input, time_input])
    x = layers.Dense(128, activation="relu")(merged_input)
    x = layers.Dense(128, activation="relu")(x)
    validity = layers.Dense(1, activation="sigmoid")(x)
    return Model([data_input, time_input], validity)

# Inicializar o Gerador e o Discriminador
stgan_generator = build_stgan_generator(input_dim, time_dim)
stgan_discriminator = build_stgan_discriminator(input_dim, time_dim)

# Compilar o Discriminador
stgan_discriminator.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Combinar STGAN
time_input = layers.Input(shape=(time_dim,))
data_input = layers.Input(shape=(input_dim,))
generated_data = stgan_generator(layers.concatenate([data_input, time_input]))
validity = stgan_discriminator([generated_data, time_input])
stgan = Model([data_input, time_input], validity)
stgan.compile(optimizer="adam", loss="binary_crossentropy")

# Treinamento do STGAN
def train_stgan(generator, discriminator, gan, data_with_gaps, conditions, mask, epochs=5000, batch_size=64):
    for epoch in range(epochs):
        # Gerar dados falsos
        noise = tf.random.normal((batch_size, input_dim))
        real_indices = np.random.choice(len(data_with_gaps), batch_size, replace=False)
        real_data = data_with_gaps[real_indices]
        real_conditions = conditions[real_indices]

        generated_data = generator.predict(np.concatenate([noise, real_conditions], axis=1), verbose=0)

        # Preparar dados para o discriminador
        X_discriminator = [np.vstack((real_data, generated_data)),
                           np.vstack((real_conditions, real_conditions))]
        y_discriminator = np.vstack((np.ones((batch_size, 1)), np.zeros((batch_size, 1))))

        # Treinar o discriminador
        discriminator.trainable = True
        discriminator_loss = discriminator.train_on_batch(X_discriminator, y_discriminator)

        # Treinar o gerador
        noise = tf.random.normal((batch_size, input_dim))
        y_generator = np.ones((batch_size, 1))
        discriminator.trainable = False
        generator_loss = gan.train_on_batch([noise, real_conditions], y_generator)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}: Generator Loss = {generator_loss}, Discriminator Loss = {discriminator_loss[0]}")

# Treinar STGAN
train_stgan(stgan_generator, stgan_discriminator, stgan, data_with_gaps, conditions, mask)

# Imputação com STGAN
noise = tf.random.normal((len(data_with_gaps), input_dim))
imputed_values = stgan_generator.predict(np.concatenate([data_with_gaps, conditions], axis=1), verbose=0)
imputed_data = data_with_gaps.copy()
imputed_data[mask.flatten() == 1] = imputed_values[mask.flatten() == 1]

# Calcular RMSE
real_missing_values = original_vazao[mask.flatten() == 1]
rmse = np.sqrt(mean_squared_error(real_missing_values, imputed_data[mask.flatten() == 1]))
print(f"RMSE: {rmse}")
